In [1]:
import pandas as pd
import sys
sys.path.append('/home/azureuser/cloudfiles/code/Users/manhductranvu/reeval-multi/mirt-official')
from load_params import load_and_rotate

resmat = pd.read_pickle("../data/resmat.pkl")

theta, a, b = load_and_rotate(rotation='varimax')

# Step 1: Get the final theta tensor into a NumPy array
theta_abilities = theta
# Step 2: Create a labeled pandas DataFrame
# Use the model names from your original resmat for the index
model_names = resmat.index
factor_names = [f'F{i+1}' for i in range(theta.shape[1])]
ability_df = pd.DataFrame(theta_abilities, index=model_names, columns=factor_names)

/Users/ronan/Developer/reeval-multi/mirt-official/load_params.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(model_path, map_location=torch.dev

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 19])
Original 'a' matrix shape: torch.Size([78712, 19])

--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 19)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 19)

--- Final Standardized Z-Scores (from transformed theta) ---
These are the scores you should use for interpretation.
[[ 1.36679121 -1.29142072  0.02368664 ...  0.5948743   1.20452549
  -1.32143699]
 [ 0.8694796   0.65897874 -2.01923406 ... -0.49557078  1.18055759
  -1.37415132]
 [ 1.82481648 -1.4122685  -0.51204663 ...  0.78741921 -0.97111116
  -1.49879246]
 ...
 [-0.9912067  -0.93205986 -0.40201418 ...  0.99411487 -0.17602767
   1.24166521]
 [-0.71577314 -0.59205097 -0.52410342 ...  1.13348854 -0.10466067
   0.66230223]
 [ 0.18720646 -0.9709694   0.55470963 ...  0.3900838   1.68143397
  -0.28299898]]


In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import ward, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

# --- Your provided code to load the a-matrix ---
# (Assuming 'a' is a NumPy array of shape (num_items, num_dimensions))
# For demonstration, let's create a sample 'a' matrix.
# Replace this with your actual 'a' matrix.
# np.random.seed(42)
# a = np.vstack([
#     np.random.rand(10, 3) * [3, 0.5, 0.5] + [-0.5, 0, 0],  # Cluster 1 (Dim 1)
#     np.random.rand(10, 3) * [0.5, 3, 0.5] + [0, -0.5, 0],  # Cluster 2 (Dim 2)
#     np.random.rand(10, 3) * [0.5, 0.5, 3] + [0, 0, -0.5]   # Cluster 3 (Dim 3)
# ])
# model_names = [f'Item_{i+1}' for i in range(a.shape[0])]
# ----------------------------------------------------

## Step 1: Calculate the Direction Cosines for each Item
# This normalizes each item's a-vector to a unit length of 1, focusing on direction.
norms = np.linalg.norm(a, axis=1, keepdims=True)
# Add a small value to avoid division by zero for items with no discrimination
direction_cosines = a / (norms + 1e-9)

## Step 2: Compute the Pairwise Angle Matrix
# The angle between two vectors is the arccosine of their dot product (since they are unit vectors).
# First, calculate the dot product between all pairs of item vectors.
dot_product_matrix = np.dot(direction_cosines, direction_cosines.T)

# Clip the values to the valid range [-1.0, 1.0] to handle potential floating-point inaccuracies.
clipped_dot_product = np.clip(dot_product_matrix, -1.0, 1.0)

# Calculate the angle in degrees. This matrix represents the "distance" or dissimilarity between items.
angle_matrix_deg = np.degrees(np.arccos(clipped_dot_product))

## Step 3: Perform Hierarchical Cluster Analysis
# Ward's method requires a condensed distance matrix (a 1D array of the upper triangle of the angle matrix).
condensed_angle_matrix = squareform(angle_matrix_deg)

# Perform the clustering using Ward's method, as recommended in the book.
linkage_matrix = ward(condensed_angle_matrix)

## Step 4: Plot the Dendrogram for Interpretation
# This visualization will show you which items cluster together.
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(18, 10))

# Use the model names from your original resmat if they are item-level
# If not, create generic labels.
if 'model_names' not in locals() or len(model_names) != a.shape[0]:
    model_names = [f'Item {i+1}' for i in range(a.shape[0])]

dendrogram(linkage_matrix,
           orientation='top',
           labels=model_names,
           distance_sort='descending',
           show_leaf_counts=True)

plt.title("Hierarchical Clustering of Test Items (Ward's Method)", fontsize=16)
plt.xlabel("Test Items", fontsize=12)
plt.ylabel("Distance (Angle Between Item Vectors in Degrees)", fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()